# Enhanced cloud masking of Sentinel-2

## Description

The default cloud mask for Sentinel-2 can be poor. Google has built a new cloud mask called 'Cloud Score'. A description of this product is availble at this [medium post](https://medium.com/google-earth/all-clear-with-cloud-score-bd6ee2e2235e)

In this notebook we show how to load the S2 data and mask it with Cloud Score


>⚠️ Note: Cloud masking with cloud score is now supported by `eemont`, see the [GEE_with_eemont](GEE_with_eemont.ipynb) notebook for details.

## Load packages

Import Python packages that are used for the analysis.


In [1]:
%matplotlib inline
import ee
import xarray as xr
import geemap as gmap

### Connect to Google Earth Engine (GEE)

In [2]:
Map = gmap.Map(center=[-35.2041, 149.2721], zoom=9)

## Load Sentinel 2 image collection


In [3]:
# Create a point with set coordinates (canberra parliament)
point = ee.Geometry.Point([149.1244, -35.3096])

def scaling_func(img):
    """
    Simple function scaling S2 bands to 0-1
    """
    return img.divide(10000)

#load the S2 product
s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                  .filterBounds(point) # filter to some region
                  .filterDate('2024-02-08', '2024-02-10')# Feb 2024
                  .select(['B4', 'B3', 'B2']) #just load the RGB bands
                  .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 50)) #cloud cover less than 50 %
                  .map(scaling_func) #divide by 10000
     )

## Load Cloud Score data

In [4]:
# Cloud Score+ image collection. Note Cloud Score+ is produced from Sentinel-2
# Level 1C data and can be applied to either L1C or L2A collections.
csPlus = ee.ImageCollection('GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED')

# Use 'cs' or 'cs_cdf', depending on your use case; see docs for guidance.
QA_BAND = 'cs'

# The threshold for masking; values between 0.50 and 0.65 generally work well.
# Higher values will remove thin clouds, haze & cirrus shadows.
CLEAR_THRESHOLD = 0.60

def masking_func(img):
    """
    Threshold the cloudscore band to create a mask
    """
    return img.updateMask(img.select(QA_BAND).gte(CLEAR_THRESHOLD))

# Make a clear median composite.
s2_masked = (s2
    .linkCollection(csPlus, [QA_BAND]) #link cloud score with S2
    .map(masking_func) # map masking function over image collection
)


## Plot

In [5]:
#after scaling, numbers are 0-1
vis_scaled = {
  'min': 0.0,
  'max': 0.3,
  'bands': ['B4', 'B3', 'B2'],
};

# add just the first image in the collection to the map
Map.addLayer(s2, vis_scaled, 'S2 no cloud-masking')
Map.addLayer(s2_masked, vis_scaled, 'Cloud masked')
Map.addLayerControl()
Map

Map(center=[-35.2041, 149.2721], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=Search…

***